<a href="https://colab.research.google.com/github/WestonWhiteUCD/PanCancer-MultiOmics/blob/main/PanCancer_MultiOmics_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PanCancer MultiOmics — Notebook 2: Preprocessing

## Purpose
Transform the aligned modalities from Notebook 1 into a clean,
model-ready fusion matrix using the correct preprocessing order:

1. Feature selection (top 500 highest-variance features per modality)
2. Imputation (mean strategy — before scaling)
3. Scaling (StandardScaler — after imputation)
4. One-hot encoding of categorical clinical variables
5. Early fusion (concatenate all modalities into one matrix)

Outputs are saved as both a CSV (for scikit-learn in Notebook 3)
and a numpy array (for PyTorch in Notebook 4).

## Preprocessing order rationale
Imputation must precede scaling because StandardScaler computes
column means and standard deviations from the data. If missing values
are not handled first, those statistics are wrong and the resulting
scaled values are meaningless.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

pd.set_option('display.max_columns', 20)
sns.set(style="whitegrid")

##1. Load Aligned Data

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/My Drive/CompSci Proj')

# Load aligned outputs from Notebook 1
aligned_path = 'aligned/'

mRNA_df     = pd.read_csv(aligned_path + 'mRNA_aligned.csv', index_col=0)
Methy_df    = pd.read_csv(aligned_path + 'Methy_aligned.csv', index_col=0)
Clinical_df = pd.read_csv(aligned_path + 'Clinical_aligned.csv', index_col=0)

print("Loaded aligned data:")
print(f"  mRNA:     {mRNA_df.shape}")
print(f"  Methy:    {Methy_df.shape}")
print(f"  Clinical: {Clinical_df.shape}")

# Quick sanity check — all three should have identical indices
assert list(mRNA_df.index) == list(Methy_df.index) == list(Clinical_df.index), \
    "Sample order mismatch between modalities!"

print("\nSanity check passed — all modalities share identical sample order.")

Mounted at /content/drive
Loaded aligned data:
  mRNA:     (8314, 3217)
  Methy:    (8314, 3139)
  Clinical: (8314, 6)

Sanity check passed — all modalities share identical sample order.


## 2. Feature Selection

In [3]:
# Separate the cancer label before any feature selection
# Label is withheld from all modeling — used post-hoc only
labels = Clinical_df['label'].copy()

N_FEATURES = 500  # top-N highest-variance features per modality

# Compute per-feature variance and select top N
mRNA_var    = mRNA_df.var(axis=0)
Methy_var   = Methy_df.var(axis=0)

mRNA_top    = mRNA_var.nlargest(N_FEATURES).index
Methy_top   = Methy_var.nlargest(N_FEATURES).index

mRNA_sel    = mRNA_df[mRNA_top]
Methy_sel   = Methy_df[Methy_top]

print("After feature selection:")
print(f"  mRNA:        {mRNA_df.shape} → {mRNA_sel.shape}")
print(f"  Methylation: {Methy_df.shape} → {Methy_sel.shape}")
print(f"  Variance range (mRNA):  "
      f"{mRNA_var[mRNA_top].min():.4f} – {mRNA_var[mRNA_top].max():.4f}")
print(f"  Variance range (Methy): "
      f"{Methy_var[Methy_top].min():.4f} – {Methy_var[Methy_top].max():.4f}")

After feature selection:
  mRNA:        (8314, 3217) → (8314, 500)
  Methylation: (8314, 3139) → (8314, 500)
  Variance range (mRNA):  1.0001 – 1.0001
  Variance range (Methy): 1.0001 – 1.0001


## 3. Preprocessing Pipeline

In [4]:
# Step 1: Impute missing values with column means
# Must come BEFORE scaling — see rationale in notebook header
imputer = SimpleImputer(strategy='mean')

mRNA_imputed  = imputer.fit_transform(mRNA_sel)
Methy_imputed = imputer.fit_transform(Methy_sel)

print("After imputation:")
print(f"  mRNA missing values:  {np.isnan(mRNA_imputed).sum()}")
print(f"  Methy missing values: {np.isnan(Methy_imputed).sum()}")

# Step 2: Scale to mean=0, std=1
# z = (x - x̄) / σ
scaler = StandardScaler()

mRNA_scaled  = scaler.fit_transform(mRNA_imputed)
Methy_scaled = scaler.fit_transform(Methy_imputed)

# Verify scaling worked correctly
print("\nAfter scaling:")
print(f"  mRNA  — mean: {mRNA_scaled.mean():.6f},  std: {mRNA_scaled.std():.6f}")
print(f"  Methy — mean: {Methy_scaled.mean():.6f},  std: {Methy_scaled.std():.6f}")

After imputation:
  mRNA missing values:  0
  Methy missing values: 0

After scaling:
  mRNA  — mean: -0.000000,  std: 1.000000
  Methy — mean: -0.000000,  std: 1.000000


### 3.1 Clinical Preprocessing - Impute + One-Hot Encode

In [5]:
# Drop label — already extracted in Cell 6, never used in modeling
Clinical_model = Clinical_df.drop(columns=['label'])

# Split into continuous and categorical columns
continuous_cols   = ['age_at_initial_pathologic_diagnosis', 'days', 'status']
categorical_cols  = ['gender', 'pathologic_stage']

# ── Step 1: Impute continuous variables with column mean ──────────────────
cont_imputer    = SimpleImputer(strategy='mean')
Clinical_cont   = cont_imputer.fit_transform(Clinical_model[continuous_cols])

# ── Step 2: Scale continuous variables ───────────────────────────────────
# CRITICAL: clinical continuous vars must be scaled BEFORE fusion
# Unscaled 'days' (range 0-10,000) would dominate PCA entirely
cont_scaler     = StandardScaler()
Clinical_cont   = cont_scaler.fit_transform(Clinical_cont)

print("After continuous imputation and scaling:")
print(f"  Missing values remaining: {np.isnan(Clinical_cont).sum()}")
print(f"  Mean (should be ~0): {Clinical_cont.mean():.6f}")
print(f"  Std  (should be ~1): {Clinical_cont.std():.6f}")

# ── Step 3: One-hot encode categorical variables ──────────────────────────
encoder         = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
Clinical_cat    = encoder.fit_transform(Clinical_model[categorical_cols])

cat_feature_names = encoder.get_feature_names_out(categorical_cols)
print(f"\nOne-hot encoded categories: {list(cat_feature_names)}")

# ── Step 4: Combine continuous + encoded categorical ──────────────────────
Clinical_processed = np.hstack([Clinical_cont, Clinical_cat])

print(f"\nClinical matrix shape after preprocessing: {Clinical_processed.shape}")
print(f"  {len(continuous_cols)} continuous + "
      f"{Clinical_cat.shape[1]} one-hot = "
      f"{Clinical_processed.shape[1]} total features")

After continuous imputation and scaling:
  Missing values remaining: 0
  Mean (should be ~0): 0.000000
  Std  (should be ~1): 1.000000

One-hot encoded categories: ['gender_FEMALE', 'gender_MALE', 'pathologic_stage_I', 'pathologic_stage_II', 'pathologic_stage_III', 'pathologic_stage_IV']

Clinical matrix shape after preprocessing: (8314, 9)
  3 continuous + 6 one-hot = 9 total features


## Early Fusion

In [6]:
# Concatenate all three preprocessed modalities horizontally
# Row i is the same patient across all three — verified in Cell 4
fusion_matrix = np.hstack([mRNA_scaled, Methy_scaled, Clinical_processed])

print("Early fusion complete:")
print(f"  mRNA:     {mRNA_scaled.shape}")
print(f"  Methy:    {Methy_scaled.shape}")
print(f"  Clinical: {Clinical_processed.shape}")
print(f"  ──────────────────────────────")
print(f"  Fusion:   {fusion_matrix.shape}")

# Final sanity checks
assert fusion_matrix.shape[0] == 8314, "Sample count mismatch!"
assert fusion_matrix.shape[1] == 1009, "Feature count mismatch!"
assert not np.isnan(fusion_matrix).any(), "NaN values detected in fusion matrix!"

print("\nAll assertions passed — fusion matrix is clean and ready for modeling.")

Early fusion complete:
  mRNA:     (8314, 500)
  Methy:    (8314, 500)
  Clinical: (8314, 9)
  ──────────────────────────────
  Fusion:   (8314, 1009)

All assertions passed — fusion matrix is clean and ready for modeling.


## 5. Save Outputs

In [7]:
output_path = 'aligned/'

# ── Save 1: Fusion matrix as CSV (for scikit-learn in Notebook 3) ─────────
fusion_df = pd.DataFrame(
    fusion_matrix,
    index=mRNA_df.index
)
fusion_df.to_csv(output_path + 'fusion_matrix.csv')
print(f"Saved fusion_matrix.csv:     {fusion_df.shape}")

# ── Save 2: Fusion matrix as numpy array (for PyTorch in Notebook 4) ──────
np.save(output_path + 'fusion_matrix.npy', fusion_matrix)
print(f"Saved fusion_matrix.npy:     {fusion_matrix.shape}")

# ── Save 3: Labels (for evaluation in Notebook 5) ─────────────────────────
labels.to_csv(output_path + 'labels.csv')
print(f"Saved labels.csv:            {labels.shape}")

# ── Save 4: Feature names (for interpretability in Notebook 5) ────────────
feature_names = (
    list(mRNA_top) +
    list(Methy_top) +
    list(cat_feature_names) +
    continuous_cols
)
pd.Series(feature_names).to_csv(output_path + 'feature_names.csv', index=False)
print(f"Saved feature_names.csv:     {len(feature_names)} features")

# Final summary
print(f"\n── Notebook 2 complete ──────────────────────────────────────────")
print(f"  Fusion matrix: {fusion_matrix.shape[0]} samples × "
      f"{fusion_matrix.shape[1]} features")
print(f"  Saved to:      {output_path}")
print(f"  Formats:       .csv (sklearn) · .npy (PyTorch)")

Saved fusion_matrix.csv:     (8314, 1009)
Saved fusion_matrix.npy:     (8314, 1009)
Saved labels.csv:            (8314,)
Saved feature_names.csv:     1009 features

── Notebook 2 complete ──────────────────────────────────────────
  Fusion matrix: 8314 samples × 1009 features
  Saved to:      aligned/
  Formats:       .csv (sklearn) · .npy (PyTorch)
